In [17]:
import os
import json
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage

# Project root
project_root = Path.cwd().resolve().parent

# Load environment variables
load_dotenv(project_root / ".env_template")

# Connect to Google Cloud Storage
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")

key_path = Path(service_account_key)
if not key_path.is_absolute():
    key_path = project_root / key_path

credentials = service_account.Credentials.from_service_account_file(
    str(key_path)
)

client = storage.Client(
    project=project_id,
    credentials=credentials
)

# Find latest census dataset
census_files = [
    blob for blob in client.list_blobs(bucket_name)
    if blob.name.endswith("_census-data.json")
]

if not census_files:
    raise FileNotFoundError("No census data found in bucket.")

latest_file = max(census_files, key=lambda blob: blob.time_created)

# Retrieve census data
census_data = json.loads(latest_file.download_as_text())
census_df = pd.DataFrame(census_data)

print("Census data retrieved successfully!")
print("File:", latest_file.name)
print("Dataset shape:", census_df.shape)

display(census_df.head())
print("\nColumns:")
print(census_df.columns.tolist())

Census data retrieved successfully!
File: 2026-10-07_07-41-55_census-data.json
Dataset shape: (245, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,NAME,B01003_001E,B01002_001E,B19013_001E,B19301_001E,B17001_001E,B17001_002E,B23025_002E,B23025_004E,B23025_005E,...,B15003_025E,B25001_001E,B25003_001E,B25003_002E,B25003_003E,B25064_001E,B25077_001E,state,county,tract
1,Census Tract 101.01; San Francisco County; Cal...,1911,37.6,89022,76257,1911,374,1266,1266,0,...,103,1433,1216,101,1115,2348,1531300,06,075,010101
2,Census Tract 101.02; San Francisco County; Cal...,1917,41.4,102826,82551,1917,155,1039,955,84,...,15,1077,930,264,666,1392,1300000,06,075,010102
3,Census Tract 102.01; San Francisco County; Cal...,2650,33.8,156017,115811,2650,181,1835,1764,23,...,84,1653,1399,406,993,3114,1927600,06,075,010201
4,Census Tract 102.02; San Francisco County; Cal...,1839,38.9,176958,140102,1814,226,1396,1295,101,...,1,1471,1099,299,800,2765,1417300,06,075,010202



Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


In [18]:
# Fix headers
census_df.columns = census_df.iloc[0]
census_df = census_df.iloc[1:].reset_index(drop=True)

# Rename columns
census_df = census_df.rename(columns={
    "NAME": "census_tract_name",
    "B01003_001E": "total_population",
    "B19013_001E": "median_household_income",
    "B17001_001E": "poverty_status_population",
    "B17001_002E": "population_below_poverty",
    "B23025_004E": "employed_population",
    "B23025_005E": "unemployed_population",
    "B15003_001E": "population_25_and_over",
    "B15003_022E": "bachelors_degree",
    "B15003_023E": "masters_degree",
    "B15003_024E": "professional_degree",
    "B15003_025E": "doctorate_degree",
    "B25003_001E": "occupied_housing_units",
    "B25003_003E": "renter_occupied_units"
})

print("Dataset shape:", census_df.shape)
display(census_df.head())

Dataset shape: (244, 32)


,census_tract_name,total_population,B01002_001E,median_household_income,B19301_001E,poverty_status_population,population_below_poverty,B23025_002E,employed_population,unemployed_population,...,doctorate_degree,B25001_001E,occupied_housing_units,B25003_002E,renter_occupied_units,B25064_001E,B25077_001E,state,county,tract
0,Census Tract 101.01; San Francisco County; Cal...,1911,37.6,89022,76257,1911,374,1266,1266,0,...,103,1433,1216,101,1115,2348,1531300,06,075,010101
1,Census Tract 101.02; San Francisco County; Cal...,1917,41.4,102826,82551,1917,155,1039,955,84,...,15,1077,930,264,666,1392,1300000,06,075,010102
2,Census Tract 102.01; San Francisco County; Cal...,2650,33.8,156017,115811,2650,181,1835,1764,23,...,84,1653,1399,406,993,3114,1927600,06,075,010201
3,Census Tract 102.02; San Francisco County; Cal...,1839,38.9,176958,140102,1814,226,1396,1295,101,...,1,1471,1099,299,800,2765,1417300,06,075,010202
4,Census Tract 103; San Francisco County; Califo...,3824,46.2,158144,93972,3824,361,2779,2655,124,...,239,2569,2060,440,1620,2417,1202500,06,075,010300


In [19]:
# Keep relevant columns
census_columns = [
    "state",
    "county",
    "tract",
    "total_population",
    "median_household_income",
    "poverty_status_population",
    "population_below_poverty",
    "employed_population",
    "unemployed_population",
    "population_25_and_over",
    "bachelors_degree",
    "masters_degree",
    "professional_degree",
    "doctorate_degree",
    "occupied_housing_units",
    "renter_occupied_units"
]

census_clean = census_df[census_columns].copy()

# Create unique census tract identifier
census_clean["GEOID"] = (
    census_clean["state"].astype(str).str.zfill(2)
    + census_clean["county"].astype(str).str.zfill(3)
    + census_clean["tract"].astype(str).str.zfill(6)
)

# Remove geographic columns already represented by GEOID
census_clean = census_clean.drop(columns=["state", "county", "tract"])

# Convert socioeconomic columns to numeric
numeric_columns = census_clean.columns.drop("GEOID")

census_clean[numeric_columns] = census_clean[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Dataset shape:", census_clean.shape)
display(census_clean.head())

Dataset shape: (244, 14)


,total_population,median_household_income,poverty_status_population,population_below_poverty,employed_population,unemployed_population,population_25_and_over,bachelors_degree,masters_degree,professional_degree,doctorate_degree,occupied_housing_units,renter_occupied_units,GEOID
0,1911,89022,1911,374,1266,0,1689,843,164,11,103,1216,1115,06075010101
1,1917,102826,1917,155,955,84,1529,382,274,113,15,930,666,06075010102
2,2650,156017,2650,181,1764,23,2240,892,659,82,84,1399,993,06075010201
3,1839,176958,1814,226,1295,101,1567,749,504,115,1,1099,800,06075010202
4,3824,158144,3824,361,2655,124,3360,1526,670,232,239,2060,1620,06075010300


In [20]:
import numpy as np

# Poverty rate
census_clean["poverty_rate"] = (
    census_clean["population_below_poverty"]
    / census_clean["poverty_status_population"].replace(0, np.nan)
) * 100

# Unemployment rate
census_clean["unemployment_rate"] = (
    census_clean["unemployed_population"]
    / (
        census_clean["employed_population"]
        + census_clean["unemployed_population"]
    ).replace(0, np.nan)
) * 100

# Bachelor's degree or higher
census_clean["bachelors_or_higher_rate"] = (
    census_clean[
        [
            "bachelors_degree",
            "masters_degree",
            "professional_degree",
            "doctorate_degree"
        ]
    ].sum(axis=1)
    / census_clean["population_25_and_over"].replace(0, np.nan)
) * 100

# Renter occupancy rate
census_clean["renter_occupancy_rate"] = (
    census_clean["renter_occupied_units"]
    / census_clean["occupied_housing_units"].replace(0, np.nan)
) * 100

# Preview calculated rates
display(
    census_clean[
        [
            "GEOID",
            "poverty_rate",
            "unemployment_rate",
            "bachelors_or_higher_rate",
            "renter_occupancy_rate"
        ]
    ].head()
)

,GEOID,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,06075010101,19.570905,0.000000,66.370634,91.694079
1,06075010102,8.085550,8.084697,51.275343,71.612903
2,06075010201,6.830189,1.287073,76.651786,70.979271
3,06075010202,12.458655,7.234957,87.364391,72.793449
4,06075010300,9.440377,4.462037,79.375000,78.640777


In [21]:
# Select final columns
final_columns = [
    "GEOID",
    "total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "bachelors_or_higher_rate",
    "renter_occupancy_rate"
]

census_final_df = census_clean[final_columns].copy()

# Replace invalid Census income values with missing values
census_final_df.loc[
    census_final_df["median_household_income"] < 0,
    "median_household_income"
] = np.nan

# Remove incomplete tracts and tracts with zero population
census_final_df = census_final_df.dropna()
census_final_df = census_final_df[
    census_final_df["total_population"] > 0
].reset_index(drop=True)

# Final verification
print("Final dataset shape:", census_final_df.shape)

print("\nMissing values:")
print(census_final_df.isna().sum())

print("\nDuplicate GEOIDs:")
print(census_final_df["GEOID"].duplicated().sum())

rate_columns = [
    "poverty_rate",
    "unemployment_rate",
    "bachelors_or_higher_rate",
    "renter_occupancy_rate"
]

print("\nRates outside 0–100%:")
print(
    ((census_final_df[rate_columns] < 0) |
     (census_final_df[rate_columns] > 100)).sum()
)

display(census_final_df.head())

Final dataset shape: (234, 7)

Missing values:
0
GEOID                       0
total_population            0
median_household_income     0
poverty_rate                0
unemployment_rate           0
bachelors_or_higher_rate    0
renter_occupancy_rate       0
dtype: int64

Duplicate GEOIDs:
0

Rates outside 0–100%:
0
poverty_rate                0
unemployment_rate           0
bachelors_or_higher_rate    0
renter_occupancy_rate       0
dtype: int64


,GEOID,total_population,median_household_income,poverty_rate,unemployment_rate,bachelors_or_higher_rate,renter_occupancy_rate
0,06075010101,1911,89022.0,19.570905,0.000000,66.370634,91.694079
1,06075010102,1917,102826.0,8.085550,8.084697,51.275343,71.612903
2,06075010201,2650,156017.0,6.830189,1.287073,76.651786,70.979271
3,06075010202,1839,176958.0,12.458655,7.234957,87.364391,72.793449
4,06075010300,3824,158144.0,9.440377,4.462037,79.375000,78.640777
